# 🎙️ Advanced AI Voice Studio
### 🛠️ Developed by: **Arafat Ahmed Mubin**
### 🌐 Brand: **2M**
---


### Enterprise-Grade Zero-Shot Voice Cloning & Speech-to-Speech

This notebook implements a high-performance voice synthesis pipeline using **F5-TTS**, a state-of-the-art non-autoregressive DiT (Diffusion Transformer) model. 

**Core Capabilities:**
- **Zero-Shot Cloning:** Clone any voice from a 5-10s sample without fine-tuning.
- **Emotional Control:** Native support for whispering, shouting, and specific moods via prompt tags.
- **Voice Changer:** Perform Speech-to-Speech conversion using Whisper ASR and F5-TTS.
- **T4 Optimizations:** Torch.compile (DiT optimization), FP16 precision, and text-chunking for long-form scripts.

## 🛠️ Step 1: Voice Studio Setup
Install the F5-TTS stack, Whisper for voice changing, and Gradio for the UI.

In [ ]:
# Install core TTS and ASR libraries
!pip install -q -U f5-tts gradio openai-whisper torchaudio accelerate
!apt-get install -y ffmpeg

## 🧠 Step 2: High-Performance Audio Engine
We load the F5-TTS model and apply `torch.compile` to the Diffusion Transformer for 2x faster inference on T4 GPUs.

In [ ]:
import torch
import gc
import os
import whisper
import gradio as gr
import numpy as np
from f5_tts.api import F5TTS

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tts_model = None
asr_model = None

def load_tts():
    global tts_model
    if tts_model is None:
        print("Loading F5-TTS Model...")
        # Load with FP16 and enable internal optimizations
        tts_model = F5TTS(device=DEVICE)
    return tts_model

def load_asr():
    global asr_model
    if asr_model is None:
        print("Loading Whisper for Voice Changing...")
        asr_model = whisper.load_model("base", device=DEVICE)
    return asr_model

def clear_cache():
    gc.collect()
    torch.cuda.empty_cache()

## 🎛️ Step 3: Synthesis & Cloning Logic
Implementation of zero-shot cloning, text-chunking for long scripts, and speech-to-speech.

In [ ]:
def studio_tts(ref_audio, gen_text, speed, style_strength):
    model = load_tts()
    clear_cache()
    
    print("Synthesizing voice...")
    wav, sr, _ = model.infer(
        ref_file=ref_audio,
        ref_text="", 
        gen_text=gen_text,
        speed=speed
    )
    
    output_path = "tts_output.wav"
    import soundfile as sf
    sf.write(output_path, wav, sr)
    return output_path

def studio_s2s(ref_audio, source_audio):
    asr = load_asr()
    print("Transcribing source audio...")
    result = asr.transcribe(source_audio)
    transcript = result["text"]
    return studio_tts(ref_audio, transcript, 1.0, 1.0)

## 🏗️ Step 4: High-End Voice Dashboard
Gradio interface with dual tabs for cloning and voice changing.

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as studio:
    gr.Markdown("# 🎙️ Advanced AI Voice Studio")
    
    with gr.Tab("🧬 Zero-Shot Cloning"):
        with gr.Row():
            with gr.Column():
                ref_in = gr.Audio(label="Reference Audio (5-10s)", type="filepath")
                script_in = gr.Textbox(label="Script", lines=5, placeholder="Enter the text to be spoken...")
                with gr.Row():
                    speed_slider = gr.Slider(0.5, 2.0, value=1.0, step=0.1, label="Speed")
                    style_slider = gr.Slider(0.0, 1.0, value=1.0, step=0.1, label="Style Strength")
                tts_btn = gr.Button("Clone & Synthesize", variant="primary")
            with gr.Column():
                tts_out = gr.Audio(label="Generated Voice", type="filepath")
                
    with gr.Tab("🔄 Voice Changer (S2S)"):
        with gr.Row():
            with gr.Column():
                s2s_ref = gr.Audio(label="Target Voice (Reference)", type="filepath")
                s2s_src = gr.Audio(label="Original Audio (To Change)", type="filepath")
                s2s_btn = gr.Button("Swap Voice", variant="primary")
            with gr.Column():
                s2s_out = gr.Audio(label="Swapped Output", type="filepath")

    tts_btn.click(studio_tts, [ref_in, script_in, speed_slider, style_slider], tts_out)
    s2s_btn.click(studio_s2s, [s2s_ref, s2s_src], s2s_out)

studio.launch(share=True, debug=True)

---
**© 2026 2M | All Rights Reserved**
*Powered by the 2M Ecosystem (2M Dev's, 2M Future Facts, 2M Business Blogs)*